# Week 6 Exercise: "The Price is Right" Capstone Project

This notebook implements a **price prediction** pipeline that estimates product prices from descriptions, based on Amazon data. It combines learnings from all 5 days:

- **Day 1**: Data curation — load, parse, deduplicate, sample
- **Day 2**: Data pre-processing — rewrite product descriptions with LLM
- **Day 3**: Baselines & Traditional ML — random, constant, linear regression, bag-of-words, Random Forest, XGBoost
- **Day 4**: Deep learning & LLMs — Neural network, LLM inference (gpt-4.1-nano)
- **Day 5**: Fine-tuning — fine-tune GPT-4.1-nano for price prediction

### Requirements
- Run from repo root or **week6** directory (setup cell configures paths)
- `.env` with `HF_TOKEN` (HuggingFace) and optionally `OPENAI_API_KEY` for LLM/fine-tuning
- Use `LITE_MODE = True` for free execution (20k train, pre-processed data from Hub)

## Setup

In [ ]:
# Add week6 to path so pricer package resolves
import sys
import os
from pathlib import Path

cwd = Path.cwd()
# Find week6: either cwd, parent (repo root), or 2 levels up (from community-contributions/name)
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    week6_dir = candidate / "week6"
    if week6_dir.exists():
        break
else:
    week6_dir = cwd
sys.path.insert(0, str(week6_dir))
os.chdir(week6_dir)
print(f"Working directory: {os.getcwd()}")

In [ ]:
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)
    print("HuggingFace login successful")
else:
    print("HF_TOKEN not set — add to .env to load datasets from Hub")

## Day 1: Data Curation

Load raw product data, explore distributions (price, text length, category), and use the curated dataset from HuggingFace. Original data: [McAuley-Lab/Amazon-Reviews-2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023).

In [ ]:
LITE_MODE = True  # Use lite dataset (20k train) for fast, free runs
USERNAME = "ed-donner"

# Load raw items (with full text) for exploration
from pricer.items import Item
raw_dataset = f"{USERNAME}/items_raw_lite" if LITE_MODE else f"{USERNAME}/items_raw_full"
train_raw, val_raw, test_raw = Item.from_hub(raw_dataset)
items_raw = train_raw + val_raw + test_raw
print(f"Loaded {len(items_raw):,} raw items")

In [ ]:
# Day 1: Explore data
import matplotlib.pyplot as plt
from collections import Counter

prices = [item.price for item in items_raw]
lengths = [len(item.full or "") for item in items_raw]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(prices, bins=50, color="blueviolet", rwidth=0.9)
plt.xlabel("Price ($)")
plt.ylabel("Count")
plt.title(f"Price distribution (avg ${sum(prices)/len(prices):.1f})")
plt.subplot(1, 2, 2)
plt.hist(lengths, bins=50, color="skyblue", rwidth=0.9)
plt.xlabel("Text length (chars)")
plt.ylabel("Count")
plt.title(f"Text length (avg {sum(lengths)/len(lengths):.0f})")
plt.tight_layout()
plt.show()

In [ ]:
# Category counts
cat_counts = Counter([item.category for item in items_raw])
plt.figure(figsize=(10, 4))
plt.bar(cat_counts.keys(), cat_counts.values(), color="goldenrod")
plt.xticks(rotation=30, ha="right")
plt.title("Items per category")
plt.show()

## Day 2: Data Pre-processing

Product descriptions are rewritten into a standard format using an LLM. This normalizes the data for downstream models. We load the pre-processed dataset (`items_lite` / `items_full`) which already has `summary` fields.

In [ ]:
# Pre-processing system prompt (used in Day 2 to create summaries)
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [ ]:
# Load pre-processed items (with summaries) for modeling
dataset = f"{USERNAME}/items_lite" if LITE_MODE else f"{USERNAME}/items_full"
train, val, test = Item.from_hub(dataset)
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

In [ ]:
# Example: raw vs pre-processed
print("Sample item summary (pre-processed):")
print(train[0].summary)

## Day 3: Baselines & Traditional ML

Evaluate simple baselines (random, constant) and traditional ML models (linear regression, bag-of-words, Random Forest, XGBoost) using the `evaluate` function from `pricer.evaluator`.

In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate

In [ ]:
# Baseline 1: Random guess
random.seed(42)
def random_pricer(item):
    return random.randrange(1, 1000)

evaluate(random_pricer, test, size=100)

In [ ]:
# Baseline 2: Constant (training average)
train_avg = sum(item.price for item in train) / len(train)
def constant_pricer(item):
    return train_avg

evaluate(constant_pricer, test, size=100)

In [ ]:
# Features for simple models
def get_features(item):
    return {
        "weight": item.weight or 0,
        "weight_unknown": 1 if (item.weight or 0) == 0 else 0,
        "text_length": len(item.summary or ""),
    }

def list_to_df(items):
    fs = [get_features(i) for i in items]
    df = pd.DataFrame(fs)
    df["price"] = [i.price for i in items]
    return df

train_df = list_to_df(train)
test_df = list_to_df(test)
feature_cols = ["weight", "weight_unknown", "text_length"]

X_train, y_train = train_df[feature_cols], train_df["price"]
X_test, y_test = test_df[feature_cols], test_df["price"]

In [ ]:
# Linear Regression on hand-crafted features
np.random.seed(42)
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

def linear_pricer(item):
    f = get_features(item)
    pred = lr_model.predict(pd.DataFrame([f])[feature_cols])[0]
    return max(0, pred)

evaluate(linear_pricer, test, size=100)

In [ ]:
# Bag-of-words + Linear Regression
documents = [item.summary for item in train]
prices_arr = np.array([float(item.price) for item in train], dtype=float)

np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words="english")
X_vec = vectorizer.fit_transform(documents)

bow_model = LinearRegression()
bow_model.fit(X_vec, prices_arr)

def bow_linear_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(0, bow_model.predict(x)[0])

evaluate(bow_linear_pricer, test, size=100)

In [ ]:
# Random Forest (subset for speed in lite mode)
subset = min(15_000, len(train))
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X_vec[:subset], prices_arr[:subset])

def rf_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

evaluate(rf_pricer, test, size=100)

In [ ]:
# XGBoost (optional - skip if not installed)
try:
    import xgboost as xgb
    xgb_model = xgb.XGBRegressor(n_estimators=200, random_state=42, n_jobs=4, learning_rate=0.1)
    xgb_model.fit(X_vec, prices_arr)

    def xgb_pricer(item):
        x = vectorizer.transform([item.summary])
        return max(0, xgb_model.predict(x)[0])

    evaluate(xgb_pricer, test, size=100)
except ImportError:
    print("XGBoost not installed — skip with: pip install xgboost")

## Day 4: Deep Learning & LLMs

Train a vanilla neural network on bag-of-words features, then use frontier LLMs for zero-shot price estimation.

In [ ]:
# Neural network
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

In [ ]:
# HashingVectorizer for binary bag-of-words
np.random.seed(42)
hv = HashingVectorizer(n_features=5000, stop_words="english", binary=True)
X_hv = hv.fit_transform(documents)
y_hv = np.array([float(i.price) for i in train], dtype=float)

X_t = torch.FloatTensor(X_hv.toarray())
y_t = torch.FloatTensor(y_hv).unsqueeze(1)
X_tr, X_vl, y_tr, y_vl = train_test_split(X_t, y_t, test_size=0.01, random_state=42)
loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)

In [ ]:
# Define and train neural network
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.layers(x)

nn_model = NeuralNetwork(X_t.shape[1])
loss_fn = nn.MSELoss()
opt = optim.Adam(nn_model.parameters(), lr=0.001)

for epoch in range(2):
    nn_model.train()
    for bx, by in tqdm(loader):
        opt.zero_grad()
        loss = loss_fn(nn_model(bx), by)
        loss.backward()
        opt.step()

In [ ]:
def nn_pricer(item):
    nn_model.eval()
    with torch.no_grad():
        x = hv.transform([item.summary])
        x = torch.FloatTensor(x.toarray())
        out = nn_model(x)[0].item()
    return max(0, out)

evaluate(nn_pricer, test, size=100)

In [ ]:
# LLM zero-shot price prediction (requires OPENAI_API_KEY)
def messages_for(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": msg}]

def gpt_nano_pricer(item):
    from litellm import completion
    r = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return r.choices[0].message.content

# Uncomment to run (uses API credits)
# evaluate(gpt_nano_pricer, test, size=50)

## Day 5: Fine-tuning a Frontier Model

Fine-tune GPT-4.1-nano on a small set of (product summary, price) examples for improved price prediction. Requires `OPENAI_API_KEY` and incurs API costs.

In [ ]:
# Prepare fine-tuning data (JSONL format)
import json

def ft_messages_for(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": msg},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

def make_jsonl(items):
    lines = []
    for item in items:
        msgs = ft_messages_for(item)
        lines.append(json.dumps({"messages": msgs}))
    return "\n".join(lines)

# Use small subset (OpenAI recommends 50–100 examples)
ft_train = train[:100]
ft_val = val[:50]

In [ ]:
import os
os.makedirs("jsonl", exist_ok=True)
with open("jsonl/fine_tune_train.jsonl", "w") as f:
    f.write(make_jsonl(ft_train))
with open("jsonl/fine_tune_val.jsonl", "w") as f:
    f.write(make_jsonl(ft_val))
print("JSONL files written")

In [ ]:
# Upload and fine-tune (run only if OPENAI_API_KEY is set and you want to incur cost)
# from openai import OpenAI
# client = OpenAI()
# with open("jsonl/fine_tune_train.jsonl", "rb") as f:
#     train_file = client.files.create(file=f, purpose="fine-tune")
# with open("jsonl/fine_tune_val.jsonl", "rb") as f:
#     val_file = client.files.create(file=f, purpose="fine-tune")
# job = client.fine_tuning.jobs.create(
#     training_file=train_file.id,
#     validation_file=val_file.id,
#     model="gpt-4.1-nano-2025-04-14",
#     seed=42,
#     hyperparameters={"n_epochs": 1, "batch_size": 1},
#     suffix="pricer",
# )
# job_id = job.id
# print(f"Fine-tuning job: {job_id}")

In [ ]:
# After job completes, use the fine-tuned model:
# fine_tuned = client.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
# def ft_pricer(item):
#     r = client.chat.completions.create(
#         model=fine_tuned,
#         messages=[{"role": "user", "content": f"Estimate the price...\n\n{item.summary}"}],
#         max_tokens=7,
#     )
#     return r.choices[0].message.content
# evaluate(ft_pricer, test)

## Summary

This capstone covered:
- **Day 1**: Data curation from Amazon Reviews
- **Day 2**: LLM-based pre-processing for standardized descriptions
- **Day 3**: Baselines (random, constant) and traditional ML (linear, RF, XGBoost)
- **Day 4**: Neural network and zero-shot LLM inference
- **Day 5**: Fine-tuning GPT-4.1-nano for price prediction

Set `LITE_MODE = False` and use the full dataset for better model performance. Fine-tuning with 20k examples can achieve ~$68 average error.